# Import Libraries

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

# Load and Preprocess Dataset

In [ ]:
data = """machine learning models learn patterns from data.
sequence models process data step by step.
recurrent neural networks are designed for sequential tasks.
rnn models maintain hidden states across time steps.

long short term memory networks solve long dependency problems.
lstm uses gates to control information flow.
gru models simplify the lstm architecture.
sequence prediction is useful in many applications.

language modeling predicts the next word in a sentence.
speech recognition processes audio sequences.
time series forecasting predicts future values.
music generation creates new melodies.

generative models learn probability distributions.
they generate new samples similar to training data.
sequence generation is widely used in artificial intelligence.
deep learning improves sequence modeling performance."""

# Tokenization and Numerical Representation

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([data])

total_words = len(tokenizer.word_index) + 1
print("Total words:", total_words)

# Create Input-Output Sequence Pairs

In [ ]:
input_sequences = []

for line in data.split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_seq = token_list[:i+1]
        input_sequences.append(n_gram_seq)

max_seq_len = max(len(seq) for seq in input_sequences)

input_sequences = pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre')

X = input_sequences[:, :-1]
y = input_sequences[:, -1]

y = tf.keras.utils.to_categorical(y, num_classes=total_words)

# Build LSTM Model

In [ ]:
model = Sequential([
    Embedding(total_words, 64, input_length=max_seq_len-1),
    LSTM(100),
    Dense(total_words, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

# Train the Model

In [ ]:
model.fit(X, y, epochs=200, verbose=1)

# Generate New Sequences

In [ ]:
def generate_text(seed_text, next_words):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_seq_len-1, padding='pre')
        predicted = np.argmax(model.predict(token_list), axis=-1)[0]

        for word, index in tokenizer.word_index.items():
            if index == predicted:
                seed_text += " " + word
                break
    return seed_text

print(generate_text("machine learning", 5))
print(generate_text("sequence models", 5))

# Component II: Transformer Based Sequence Generation

# Tokenization

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, Dense, LayerNormalization, Dropout, MultiHeadAttention, GlobalAveragePooling1D, Input
from tensorflow.keras.models import Model

tokenizer = Tokenizer()
tokenizer.fit_on_texts(data.split('\n'))

total_words = len(tokenizer.word_index) + 1

# Positional Encoding

In [ ]:
input_sequences = []

for line in data.split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        input_sequences.append(token_list[:i+1])

max_len = max(len(seq) for seq in input_sequences)

input_sequences = pad_sequences(input_sequences, maxlen=max_len, padding='pre')

X = input_sequences[:, :-1]
y = input_sequences[:, -1]

# Transformer Model

In [ ]:
def transformer_block(inputs, head_size, num_heads, ff_dim, dropout=0.1):
    x = MultiHeadAttention(num_heads=num_heads, key_dim=head_size)(inputs, inputs)
    x = Dropout(dropout)(x)
    x = LayerNormalization(epsilon=1e-6)(x + inputs)

    ffn = Dense(ff_dim, activation="relu")(x)
    ffn = Dense(inputs.shape[-1])(ffn)
    ffn = Dropout(dropout)(ffn)

    return LayerNormalization(epsilon=1e-6)(x + ffn)

# Build Transformer Model

In [ ]:
embed_dim = 64
num_heads = 2
ff_dim = 128

inputs = Input(shape=(max_len - 1,))
x = Embedding(total_words, embed_dim)(inputs)

x = transformer_block(x, embed_dim, num_heads, ff_dim)
x = GlobalAveragePooling1D()(x)

outputs = Dense(total_words, activation="softmax")(x)

model = Model(inputs, outputs)
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

model.summary()

# Prepare Data for Transformer

In [ ]:
model.fit(X, y, epochs=200, verbose=1)

# Train Transformer Model

In [ ]:
transformer_model.fit(X_t, y_t, epochs=200, verbose=1)

# Generate Sequences using Transformer

In [ ]:
index_to_word = {v: k for k, v in tokenizer.word_index.items()}

In [ ]:
def sample_with_temperature(preds, temperature=0.8):
    preds = np.asarray(preds).astype('float64')
    preds = np.log(preds + 1e-9) / temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    return np.random.choice(len(preds), p=preds)

In [ ]:
def generate_text(seed_text, next_words=5, temperature=0.8):
    for _ in range(next_words):
        seq = tokenizer.texts_to_sequences([seed_text])[0]
        seq = pad_sequences([seq], maxlen=max_len - 1, padding='pre')

        preds = model.predict(seq, verbose=0)[0]
        pred_index = sample_with_temperature(preds, temperature)

        if pred_index == 0:
            continue

        next_word = index_to_word.get(pred_index)

        if next_word is None:
            break

        seed_text += " " + next_word

    return seed_text

In [ ]:
print(generate_text("deep learning"))
print(generate_text("sequence models"))
print(generate_text("transformers are"))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/GenAI/Lab 9/"

In [ ]:
# Save LSTM model

model.save(BASE_DIR+"lstm_sequence_model.h5")

# Save Transformer model
transformer_model.save(BASE_DIR+"transformer_sequence_model.h5")

In [ ]:
import pickle
with open(BASE_DIR+"tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)